<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

All dependencies present.


In [78]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

Seed set


In [79]:
seed_val = 42
set_seed(seed_val)

In [80]:
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/Saved_variables/05-09-2024_all_images-20240509T194653Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated Images-20240523T233205Z-001.zip"
# !unzip "/content/drive/Shareddrives/Summer_2023_VCell_Ml/2024-05-09 New Work/New Image Generation/Generated_Images_7-2-24-20240702T205017Z-001.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular.zip"
# !unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse.zip"
!powershell -Command "Expand-Archive -Path './content/ImageJ_data/Inverse_resized.zip' -DestinationPath './content/ImageJ_data'" 

#!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Regular_resized.zip"

!powershell -Command "Expand-Archive -Path './content/ImageJ_data/Regular_resized.zip' -DestinationPath './content/ImageJ_data'" 

#!unzip "/content/drive/MyDrive/2024_Summer_Work_(Shared_Drive_Is_Full)/As of 2024-08-22 Saved variables and data/Real_Images/2024-08-22_ImageJ_Data/Inverse_resized.zip"
!echo "Unzip Complete"

ExpandArchiveHelper : Failed to create file 'C:\Users\user.SR7872\Documents\Git
hub\PatternsFormation\Working\content\ImageJ_data\Inverse_resized\bHLH2_OE_(gre
en)_(0-141_thresh.tif.csv' while expanding the archive file 'C:\Users\user.SR78
72\Documents\Github\PatternsFormation\Working\content\ImageJ_data\Inverse_resiz
ed.zip' contents as the file 'C:\Users\user.SR7872\Documents\Github\PatternsFor
mation\Working\content\ImageJ_data\Inverse_resized\bHLH2_OE_(green)_(0-141_thre
sh.tif.csv' already exists. Use the -Force parameter if you want to overwrite 
the existing directory 'C:\Users\user.SR7872\Documents\Github\PatternsFormation
\Working\content\ImageJ_data\Inverse_resized\bHLH2_OE_(green)_(0-141_thresh.tif
.csv' contents when expanding the archive file.
At C:\WINDOWS\system32\WindowsPowerShell\v1.0\Modules\Microsoft.PowerShell.Arch
ive\Microsoft.PowerShell.Archive.psm1:397 char:17
+ ...             ExpandArchiveHelper $resolvedSourcePaths $resolvedDestina ...
+                 ~~~~~

"Unzip Complete"


In [81]:
with open("./content/2024-08-18_curr_df.pkl", 'rb') as f:
  feats_df = pickle.load(f) # deserialize using load()

feats_df.head()
#feats_df = feats_df.drop([700,1159,4061], axis = "rows")
#feats_df.reset_index(inplace = True, drop = True)
#feats_df["class"] = np.array(feats_df["class"].astype(int) -1).astype(int)
#print(np.unique(np.array(feats_df["class"]), return_counts = True))

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,1,-31.683971,-14.198348


In [82]:
with open("./content/2024-08-19_feat_array_unscaled.pkl", 'rb') as f:
  feat_array_classified = pickle.load(f) # deserialize using load()

with open("./content/2024-08-18_curr_df.pkl", 'rb') as f:
  df_classes_classified = pickle.load(f) # deserialize using load()

In [83]:
with open("./content/2024-08-19_feats_df_narrow.pkl", 'rb') as f:
  feats_df_new = pickle.load(f) # deserialize using load()

feats_df.shape

(32210, 8)

In [84]:
df_classes_classified.head()

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,1,-31.683971,-14.198348


In [85]:
feats_df.head()

,class,noise,path,seed,dir,pred_class,pc1,pc2
0,2,2,0.png,0,/content/Images3/,1,-31.683971,-14.198348
1,None,2,1.png,1,/content/Images3/,2,31.977729,-17.444531
2,2,2,2.png,2,/content/Images3/,1,-31.683971,-14.198348
3,None,2,3.png,3,/content/Images3/,2,31.347501,-16.048195
4,2,2,4.png,4,/content/Images3/,1,-31.683971,-14.198348


In [86]:
# Sets pred-class column equal to class column
feats_df["pred_class"] = feats_df["class"]

In [87]:
# two dataframes 
df1 = df_classes_classified[["noise","path","seed","dir","pred_class"]]
df2 = feats_df[["noise","path","seed","dir","pred_class"]]

# concat's predicted and manual classified dataframes (df_classes_classified + feats_df)
pred_and_manual_df = pd.concat([df1, df2], axis=0)

pred_and_manual_df.reset_index(inplace = True, drop = True)
feats_df_new.reset_index(inplace = True, drop = True)

In [88]:
#add full path to each dataframe to allow matching based on unique path
# df with all params: feats_df_new
# df with predicted classes: pred_and_manual_df

fdf_full_path = []
for i in range(feats_df_new.shape[0]):
  fdf_full_path.append(feats_df_new["dir"][i] + feats_df_new["path"][i])


cdf_full_path = []
for j in range(pred_and_manual_df.shape[0]):
  cdf_full_path.append((pred_and_manual_df["dir"][j] + pred_and_manual_df["path"][j]))


feats_df_new["full_path"] = fdf_full_path
pred_and_manual_df["full_path"] = cdf_full_path

In [89]:
indices_1 = []
indices_2 = []
errors = 0
counter = 0
for i in range(feats_df_new.shape[0]):
  #pull up path to search for
  path = pred_and_manual_df["full_path"][i]

  #find index within feats_df that matches path
  ind_df = feats_df_new[feats_df_new["full_path"] == path]

  if ind_df.shape[0] == 1:
    ind = ind_df.index[0]
    # print(ind_df.head())
    indices_1.append(ind)
    indices_2.append(i)
  else:
    # indices.append("error")
    # print("error")
    errors += 1


  if counter % 1000 == 0:
    print(counter)

  counter += 1




0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000


In [90]:
# match dfs

feats_df_new = feats_df_new.iloc[indices_1]
pred_and_manual_df = pred_and_manual_df.iloc[indices_2]
feats_df_new.reset_index(inplace = True, drop = True)
pred_and_manual_df.reset_index(inplace = True, drop = True)

In [91]:
same = list(np.array(feats_df_new["full_path"]) == np.array(pred_and_manual_df["full_path"]))

print(same.count(True))
print(same.count(False))
print(len(same))

28659
0
28659


In [92]:
feats_df_new["classifier_pred_class"] = pred_and_manual_df["pred_class"]

In [93]:
#its a feats_df of constrained parameters
feats_df_new.head()

feats_df = feats_df_new

In [94]:
feats_df.head()

,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,...,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,full_path,classifier_pred_class
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,...,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0,/content/Images3/1.png,2
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0,/content/Images3/3.png,2
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.982,0.0,/content/Images3/7.png,2
3,0.035847,0.078762,0.07823,0.101485,-0.115752,0.010542,0.933894,1,2,12.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.969,0.0,/content/Images3/12.png,2
4,0.029837,0.067296,0.079054,0.095099,-0.132865,0.009721,0.822421,1,2,13.png,...,200.0,0.0,1.000,0.0,1.000,0.0,0.968,0.0,/content/Images3/13.png,2


In [95]:
print(feats_df.columns[13:111])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

XGBOOST

In [96]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
! pip install graphviz
! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

In [97]:
np.unique(feats_df["num_spots"].astype(float))

array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
        11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
        22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
        33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
        44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
        55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
        66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
        77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
        88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
        99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
       110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
       121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
       132., 133., 134., 135., 136., 137., 138., 139., 140., 141., 142.,
       143., 144., 145., 146., 147., 148., 149., 15

In [98]:
# scale data

## X (independent variable) --> the feature values themselves
X = feats_df[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split



# df_train = feats_df.sample(frac = 0.80, random_state = 42)
# df_train.reset_index(inplace = True, drop = True)
# df_val_test = feats_df.drop(df_train.index)

# p_out = 1

# df_val_test = df_val_test.sample(frac = p_out, random_state = 42)
# df_val = df_val_test.sample(frac = 0.5, random_state = 42)
# df_test = df_val_test.drop(df_val.index)

# df_val.reset_index(inplace = True, drop = True)
# df_test.reset_index(inplace = True, drop = True)

In [104]:
y_test

,Ua,Ui,Ga,Gi,Da,Di,Ba
20241,0.036670,0.073226,0.040154,0.061430,0.009830,0.850876,-0.101614
14778,0.010814,0.077545,0.084799,0.106047,0.011972,0.344742,-0.120386
25231,0.031524,0.146835,0.162371,0.089494,0.009966,0.948477,-0.151192
26948,0.032629,0.072921,0.088104,0.153645,0.009018,0.894565,-0.164564
12272,0.011581,0.058755,0.142204,0.123917,0.011563,1.160790,-0.115591
...,...,...,...,...,...,...,...
13827,0.008961,0.052993,0.128148,0.147533,0.010953,0.754449,-0.150162
2855,0.019784,0.072723,0.137530,0.070223,0.011001,0.669717,-0.102735
16710,0.019573,0.059808,0.089377,0.160549,0.010973,0.480674,-0.144362
25107,0.038058,0.123075,0.108594,0.162017,0.008668,0.761952,-0.113683


In [99]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:50:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:49: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:50:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user.SR7872\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:50:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "num_boost_round" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None, ...)

In [106]:
# make predictions
y_pred = model.predict(X_test)

,0,1,2,3,4,5,6
0,0.033395,0.087170,0.064362,0.110597,0.010077,0.807379,-0.129942
1,0.011164,0.086442,0.089399,0.117670,0.010302,0.697959,-0.130809
2,0.024965,0.140235,0.125021,0.088071,0.010001,0.869328,-0.101000
3,0.035866,0.078264,0.087985,0.120647,0.009795,0.673017,-0.159450
4,0.024965,0.140235,0.125021,0.088071,0.010001,0.869328,-0.101000
...,...,...,...,...,...,...,...
5727,0.011736,0.069977,0.117183,0.112757,0.009499,1.038847,-0.129972
5728,0.024965,0.140235,0.125021,0.088071,0.010001,0.869328,-0.101000
5729,0.033395,0.087170,0.064362,0.110597,0.010077,0.807379,-0.129942
5730,0.038365,0.133653,0.101905,0.119501,0.009474,0.620962,-0.141445


In [109]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

(5732, 7)
(5732, 7)
[32.13409601 54.19867644 25.56774297 40.13796377  8.68439102 37.65799152
 33.26148726]


In [110]:
# make a dataframe with the names and feat_dfs and inverse_feat_dfs:
real_imj_df = {"path":[],"feats_df":[],"inverse_feats_df":[]}

reg_paths = os.listdir("./content/ImageJ_data/Inverse_resized/")

for path in reg_paths:
  real_imj_df["path"].append(path)
  real_imj_df["feats_df"].append(pd.read_csv("./content/ImageJ_data/Regular_resized/" + path))
  real_imj_df["inverse_feats_df"].append(pd.read_csv("./content/ImageJ_data/Inverse_resized/" + path))

real_df = pd.DataFrame(real_imj_df)

In [111]:
real_df["feats_df"][0]

,,Label,Area,Mean,StdDev,Mode,Min,Max,X,Y,...,Kurt,%Area,RawIntDen,FeretX,FeretY,FeretAngle,MinFeret,AR,Round,Solidity
0,1,bHLH2_OE_(green)_(0-141_thresh.tif,40000,67.001,112.234,0,0,255,100.000,100.000,...,-0.838,26.275,2680050,0,0,135.000,200.000,1.000,1.000,1.000
1,2,bHLH2_OE_(green)_(0-141_thresh.tif,629,255.000,0.000,255,255,255,25.869,18.262,...,NaN,100.000,160395,0,0,130.236,29.931,2.465,0.406,0.479
2,3,bHLH2_OE_(green)_(0-141_thresh.tif,34,255.000,0.000,255,255,255,68.500,2.000,...,NaN,100.000,8670,64,0,161.565,5.000,1.939,0.516,0.872
3,4,bHLH2_OE_(green)_(0-141_thresh.tif,73,255.000,0.000,255,255,255,99.856,3.253,...,NaN,100.000,18615,94,5,22.620,9.000,1.485,0.673,0.864
4,5,bHLH2_OE_(green)_(0-141_thresh.tif,550,255.000,0.000,255,255,255,130.080,29.987,...,NaN,100.000,140250,128,0,100.886,23.747,2.644,0.378,0.566
5,6,bHLH2_OE_(green)_(0-141_thresh.tif,377,255.000,0.000,255,255,255,172.792,9.720,...,NaN,100.000,96135,157,0,164.249,23.000,1.875,0.533,0.546
6,7,bHLH2_OE_(green)_(0-141_thresh.tif,12,255.000,0.000,255,255,255,196.083,0.917,...,NaN,100.000,3060,192,0,159.444,3.000,3.884,0.257,0.585
7,8,bHLH2_OE_(green)_(0-141_thresh.tif,128,255.000,0.000,255,255,255,71.633,17.148,...,NaN,100.000,32640,70,8,96.340,10.000,1.760,0.568,0.892
8,9,bHLH2_OE_(green)_(0-141_thresh.tif,1338,255.000,0.000,255,255,255,103.688,58.886,...,NaN,100.000,341190,84,10,105.051,33.470,5.638,0.177,0.448
9,10,bHLH2_OE_(green)_(0-141_thresh.tif,40,255.000,0.000,255,255,255,197.000,33.000,...,NaN,100.000,10200,195,29,116.565,6.000,1.368,0.731,0.930


In [112]:
# Add regular features

# first list is mean, second list is std except for mean and median
feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}

# feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    # if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      # feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      feats[feat].append(fdf[feat[:]][0])
      # feats[feat].append(fdf[feat][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counting done")

0
Counting done


In [113]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

# feats_df["num_spots_inverted"] = num_spots
real_df["num_spots"] = num_spots

In [114]:
for key in list(feats.keys()):
  # if key in ['Mean_inverted','Median_inverted']:
  if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    # feats_df[key + "_inverted_mean"] = feats[key][0]
    # feats_df[key + "_inverted_std"] = feats[key][1]
    real_df[key + "_mean"] = feats[key][0]
    real_df[key + "_std"] = feats[key][1]

In [67]:
# Add regular features

# first list is mean, second list is std except for mean and median
# feats = {'Mean':[],'Median':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
#          'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
#        'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
#        'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
#        'Round':[[],[]], 'Solidity':[[],[]]}

feats = {'Mean_inverted':[],'Median_inverted':[],'Area':[[],[]], 'X':[[],[]], 'Y':[[],[]], 'Perim.':[[],[]], 'BX':[[],[]],
         'BY':[[],[]], 'Width':[[],[]], 'Height':[[],[]], 'Major':[[],[]], 'Minor':[[],[]],
       'Angle':[[],[]], 'Circ.':[[],[]], 'Feret':[[],[]], 'IntDen':[[],[]], '%Area':[[],[]],
       'RawIntDen':[[],[]], 'FeretX':[[],[]], 'FeretY':[[],[]], 'FeretAngle':[[],[]], 'MinFeret':[[],[]], 'AR':[[],[]],
       'Round':[[],[]], 'Solidity':[[],[]]}


counter = 0

# iterate through all sims
for n in range(real_df.shape[0]): # grab the feats dataframe for this sim
  fdf = real_df["inverse_feats_df"][n] #iterate through all feats within the feats dictionary

  for i in range(len(feats.keys())): # select the current feat
    feat = list(feats.keys())[i]

    if feat in ['Mean_inverted','Median_inverted']: # check for feats that should be added from the overall measurement (first row)
    # if feat in ['Mean','Median']: # check for feats that should be added from the overall measurement (first row)
      feats[feat].append(fdf[feat[0:-9]][0]) #inverse
      # feats[feat].append(fdf[feat[:]][0])

    else:
      if fdf.shape[0] <= 1: # check if the fdf is only one measurement (empty result)
        feats[feat][0].append(np.mean(np.array(fdf[feat][0]))) # mean of sole measurement
        feats[feat][1].append(np.std(np.array(fdf[feat][0]))) #std of sole measurement (0)

      else:
        feats[feat][0].append(np.mean(np.array(fdf[feat][1:]))) # add mean for each of the rest of the feats using the remaining rows
        feats[feat][1].append(np.std(np.array(fdf[feat][1:]))) # add std for each of the rest of the feats using the remaining rows

  if counter % 1000 == 0:
    print(counter)
  counter += 1

print("Counter complete")

0
Counter complete


In [68]:
num_spots = []

for i in range(real_df.shape[0]):
  fdf = real_df["inverse_feats_df"][i] #iterate through all feats within the feats dictionary
  if fdf.shape[0] <= 1:
    num_spots.append(0)
  else:
    num_spots.append(fdf.shape[0]-1)

real_df["num_spots_inverted"] = num_spots
# real_df["num_spots"] = num_spots

In [69]:
for key in list(feats.keys()):
  if key in ['Mean_inverted','Median_inverted']:
  # if key in ['Mean','Median']:
    real_df[key] = feats[key]
  else:
    real_df[key + "_inverted_mean"] = feats[key][0]
    real_df[key + "_inverted_std"] = feats[key][1]
    # real_df[key + "_mean"] = feats[key][0]
    # real_df[key + "_std"] = feats[key][1]

In [70]:
print(real_df.columns[3:101])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

In [71]:
# select features of real images
real_feats = real_df[real_df.columns[3:101]]

#scale data
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(real_feats)
real_feats_scaled = scaling.transform(real_feats)



# predict params of images from real feats
y_pred = model.predict(real_feats_scaled)

real_df["pred_params"] = list(y_pred)

In [72]:
real_df["pred_params"][0]

array([ 0.02614826,  0.06258593,  0.12087097,  0.08383078,  0.01085906,
        0.85632795, -0.09742121], dtype=float32)

In [73]:
real_df["pred_params"][0]

array([ 0.02614826,  0.06258593,  0.12087097,  0.08383078,  0.01085906,
        0.85632795, -0.09742121], dtype=float32)

In [74]:
for i in range(real_df.shape[0]):
  print(real_df["path"][i])
  print(real_df["pred_params"][i])

bHLH2_OE_(green)_(0-141_thresh.tif.csv
[ 0.02614826  0.06258593  0.12087097  0.08383078  0.01085906  0.85632795
 -0.09742121]
bHLH2_RNAi_(green)_(0-170).tif.csv
[ 0.02351721  0.05714148  0.07845976  0.1292734   0.01027878  0.76815784
 -0.08877929]
F2_260_green)_(0-105).tif.csv
[ 0.02331874  0.06008431  0.12086973  0.11648673  0.00954962  0.70141786
 -0.0975161 ]
F2_A 8_(green)_(0-100).tif.csv
[ 0.02924089  0.06581408  0.1192591   0.10536818  0.00979359  0.6629838
 -0.06067621]
LF10_11-20-23_(green)_(0-152).tif.csv
[ 0.0221226   0.10830527  0.12630259  0.11860634  0.0105213   0.6667085
 -0.10096822]
MLC_F1_11-20-23_(green)_(0-105).tif.csv
[ 0.02128733  0.12389465  0.09693768  0.12160463  0.01039962  0.81738204
 -0.09693675]
mpar_rto(c-c)_(green)_(0-137).tif.csv
[ 0.02338564  0.11086314  0.10433803  0.11101844  0.01052789  0.75643164
 -0.08920884]
mpar_rto_crispr_(green)_(0-155).tif.csv
[ 0.01959674  0.11570201  0.12073527  0.10527825  0.00973131  0.9733093
 -0.08260424]
mpar_wt_(green)_